# Phase 1: Data Acquisition for Twi/Akan Voice Technology

This notebook downloads and processes the following datasets:
1. **BibleTTS** - High-quality TTS dataset with Akuapem Twi and Asante Twi
2. **Financial Inclusion Speech Dataset** - Ghana-specific financial domain speech data
3. **Mozilla Common Voice** - Community-contributed speech data
4. **AfriSpeech-200** - Large-scale African speech dataset

Compatible with both Google Colab and Kaggle environments.

## 1. Environment Setup and Dependencies

In [ ]:
%%capture
# Install required packages
!pip install -q datasets librosa soundfile tqdm pydub
!pip install -q gdown wget
!apt-get install -qq ffmpeg

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile
import tarfile
import shutil
import requests
from tqdm.auto import tqdm
import subprocess
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import librosa
import soundfile as sf
from datasets import load_dataset, Dataset, DatasetDict

# Check environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules

print(f"Environment: {'Google Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Python version: {sys.version}")

In [ ]:
# Mount Google Drive (if in Colab)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/voice-lab-data')
elif IN_KAGGLE:
    BASE_DIR = Path('/kaggle/working/voice-lab-data')
else:
    BASE_DIR = Path('./data')

# Create directory structure
DIRS = {
    'raw': BASE_DIR / 'raw',
    'processed': BASE_DIR / 'processed',
    'synthetic': BASE_DIR / 'synthetic',
    'metadata': BASE_DIR / 'metadata'
}

for dir_path in DIRS.values():
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print("Directory structure created:")
for name, path in DIRS.items():
    print(f"  {name}: {path}")

## 2. Download BibleTTS Dataset

In [ ]:
def download_file(url: str, dest_path: Path, chunk_size: int = 8192) -> bool:
    """Download file with progress bar."""
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        with open(dest_path, 'wb') as file:
            with tqdm(total=total_size, unit='iB', unit_scale=True, desc=dest_path.name) as pbar:
                for chunk in response.iter_content(chunk_size=chunk_size):
                    if chunk:
                        file.write(chunk)
                        pbar.update(len(chunk))
        return True
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False

In [ ]:
# BibleTTS URLs from OpenSLR
BIBLETTS_URLS = {
    'akuapem_twi': [
        'https://www.openslr.org/resources/129/akuapem-twi.tgz',
        'https://us.openslr.org/resources/129/akuapem-twi.tgz',  # US mirror
        'https://cn.openslr.org/resources/129/akuapem-twi.tgz'   # China mirror
    ],
    'asante_twi': [
        'https://www.openslr.org/resources/129/asante-twi.tgz',
        'https://us.openslr.org/resources/129/asante-twi.tgz',  # US mirror
        'https://cn.openslr.org/resources/129/asante-twi.tgz'   # China mirror
    ]
}

def download_bibletts():
    """Download BibleTTS datasets for Twi languages."""
    bibletts_dir = DIRS['raw'] / 'bibletts'
    bibletts_dir.mkdir(exist_ok=True)
    
    for lang, urls in BIBLETTS_URLS.items():
        file_path = bibletts_dir / f"{lang}.tgz"
        extract_dir = bibletts_dir / lang
        
        # Check if already extracted by looking for actual content
        # BibleTTS structure typically has train/dev/test folders or audio files
        is_extracted = False
        if extract_dir.exists():
            # Check for typical BibleTTS subdirectories or audio files
            has_splits = any((extract_dir / split).exists() for split in ['train', 'dev', 'test', 'valid'])
            has_audio = any(extract_dir.rglob('*.flac')) or any(extract_dir.rglob('*.wav'))
            
            # Also check if the extraction created a nested folder (e.g., akuapem-twi/akuapem-twi/)
            nested_dir = extract_dir / lang.replace('_', '-')
            has_nested = nested_dir.exists() and nested_dir.is_dir()
            
            is_extracted = has_splits or has_audio or has_nested
            
            if is_extracted:
                print(f"✓ {lang} already extracted")
                
                # If there's a nested directory, move contents up one level
                if has_nested and not has_splits and not has_audio:
                    print(f"  Fixing nested directory structure for {lang}...")
                    import shutil
                    for item in nested_dir.iterdir():
                        shutil.move(str(item), str(extract_dir / item.name))
                    nested_dir.rmdir()
                    print(f"  ✓ Fixed directory structure")
                continue
        
        # Download if not exists
        if not file_path.exists():
            print(f"\nDownloading {lang} BibleTTS...")
            for url in urls:
                if download_file(url, file_path):
                    break
            else:
                print(f"Failed to download {lang} from all mirrors")
                continue
        else:
            print(f"Archive {lang}.tgz exists, extracting...")
        
        # Extract
        print(f"Extracting {lang}...")
        try:
            with tarfile.open(file_path, 'r:gz') as tar:
                # Extract to bibletts_dir first
                tar.extractall(bibletts_dir)
            
            # Check if extraction created the expected directory
            if not extract_dir.exists():
                # Sometimes the archive extracts to a different name (e.g., akuapem-twi instead of akuapem_twi)
                alt_name = lang.replace('_', '-')
                alt_dir = bibletts_dir / alt_name
                if alt_dir.exists():
                    print(f"  Renaming {alt_name} to {lang}")
                    alt_dir.rename(extract_dir)
            
            print(f"✓ {lang} extracted successfully")
            
            # Optionally delete archive to save space
            # file_path.unlink()
            
        except Exception as e:
            print(f"Error extracting {lang}: {e}")
            continue

# Download BibleTTS
download_bibletts()

## 3. Download Financial Inclusion Dataset (Actual Data)

In [ ]:
def download_financial_inclusion_full():
    """Download actual Financial Inclusion Speech Dataset from S3.
    
    Strategy: Focus on Twi language family first (Akuapem, Asante, Fante)
    Save Ga for future phases as it's a different language family.
    """
    fi_dir = DIRS['raw'] / 'financial_inclusion'
    fi_dir.mkdir(exist_ok=True)
    
    # Check if already downloaded
    if (fi_dir / 'download_complete.flag').exists():
        print("✓ Financial Inclusion dataset already downloaded")
        return fi_dir
    
    print("Downloading Financial Inclusion Speech Dataset (Actual Data)...")
    print("Strategy: Focusing on Twi language family for better model performance")
    print("-" * 60)
    
    # URLs for Twi language family only (single-language strategy)
    FISD_URLS = {
        # Akuapem Twi
        'akuapem_twi_90': 'https://fisd-dataset.s3.amazonaws.com/fisd-akuapim-twi-90p.zip',
        'akuapem_twi_10': 'https://fisd-dataset.s3.amazonaws.com/fisd-akuapim-twi-10p.zip',
        
        # Asante Twi
        'asante_twi_90': 'https://fisd-dataset.s3.amazonaws.com/fisd-asanti-twi-90p.zip',
        'asante_twi_10': 'https://fisd-dataset.s3.amazonaws.com/fisd-asanti-twi-10p.zip',
        
        # Fante (closely related to Twi)
        'fante_90': 'https://fisd-dataset.s3.amazonaws.com/fisd-fanti-90p.zip',
        'fante_10': 'https://fisd-dataset.s3.amazonaws.com/fisd-fanti-10p.zip',
        
        # GA LANGUAGE - SAVED FOR PHASE 5+ (different language family)
        # Uncomment these lines when ready to work with Ga
        # 'ga_90': 'https://fisd-dataset.s3.amazonaws.com/fisd-ga-90p.zip',
        # 'ga_10': 'https://fisd-dataset.s3.amazonaws.com/fisd-ga-10p.zip',
    }
    
    # Track download statistics
    download_stats = {
        'total_files': len(FISD_URLS),
        'downloaded': 0,
        'extracted': 0,
        'failed': []
    }
    
    for name, url in FISD_URLS.items():
        zip_path = fi_dir / f"{name}.zip"
        extract_dir = fi_dir / name
        
        # Skip if already extracted
        if extract_dir.exists() and any(extract_dir.iterdir()):
            print(f"✓ {name} already extracted")
            download_stats['extracted'] += 1
            continue
        
        # Download if not exists
        if not zip_path.exists():
            print(f"\nDownloading {name}...")
            print(f"  URL: {url}")
            
            if download_file(url, zip_path):
                download_stats['downloaded'] += 1
                print(f"✓ Downloaded {name}")
            else:
                print(f"✗ Failed to download {name}")
                download_stats['failed'].append(name)
                continue
        
        # Extract
        try:
            print(f"Extracting {name}...")
            extract_dir.mkdir(exist_ok=True)
            
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            
            download_stats['extracted'] += 1
            print(f"✓ Extracted {name} successfully")
            
            # Delete zip to save space (optional)
            if IN_COLAB or IN_KAGGLE:  # Save space in cloud environments
                zip_path.unlink()
                print(f"  Deleted {name}.zip to save space")
                
        except Exception as e:
            print(f"✗ Error extracting {name}: {e}")
            download_stats['failed'].append(name)
    
    # Create metadata about the download
    metadata = {
        'download_date': pd.Timestamp.now().isoformat(),
        'languages': ['akuapem_twi', 'asante_twi', 'fante'],
        'total_datasets': download_stats['extracted'],
        'failed_downloads': download_stats['failed'],
        'strategy': 'single_language_family',
        'notes': 'Focused on Twi family only. Ga saved for Phase 5+'
    }
    
    # Save metadata
    with open(fi_dir / 'dataset_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Create completion flag
    if download_stats['extracted'] > 0:
        with open(fi_dir / 'download_complete.flag', 'w') as f:
            f.write(f"Downloaded on {pd.Timestamp.now()}")
    
    # Print summary
    print("\n" + "="*60)
    print("📊 Financial Inclusion Dataset Download Summary:")
    print(f"  Total datasets: {download_stats['total_files']}")
    print(f"  Successfully downloaded: {download_stats['downloaded']}")
    print(f"  Successfully extracted: {download_stats['extracted']}")
    
    if download_stats['failed']:
        print(f"  ⚠️ Failed: {', '.join(download_stats['failed'])}")
    
    print("\n📝 Language Strategy:")
    print("  ✅ Downloaded: Akuapem Twi, Asante Twi, Fante")
    print("  ⏳ Saved for later: Ga (Phase 5+)")
    print("\n💡 Why single language family?")
    print("  - Better model accuracy (shared phonemes)")
    print("  - Faster training convergence")
    print("  - Simpler text processing")
    print("  - Can achieve <20% WER faster")
    
    return fi_dir

# Download the actual Financial Inclusion dataset
fi_dataset_path = download_financial_inclusion_full()

In [ ]:
# Analyze Financial Inclusion dataset structure
if fi_dataset_path and fi_dataset_path.exists():
    print("Financial Inclusion Dataset Structure:")
    print("="*60)
    
    # Count audio files and calculate statistics
    total_audio = 0
    total_text = 0
    dataset_info = {}
    
    for dataset_dir in fi_dataset_path.iterdir():
        if dataset_dir.is_dir() and not dataset_dir.name.startswith('.'):
            # Handle nested directory structure
            # The actual data is in subdirectories like: fante_90/fisd-fanti-90p/audios
            
            # Look for nested directories with audio files
            audio_files = []
            text_files = []
            
            # Check for nested structure (e.g., fante_90/fisd-fanti-90p/audios)
            for subdir in dataset_dir.rglob('audios'):
                if subdir.is_dir():
                    audio_files.extend(list(subdir.glob('*.wav')))
                    audio_files.extend(list(subdir.glob('*.mp3')))
            
            # Also check for text/transcript files
            for subdir in dataset_dir.rglob('texts'):
                if subdir.is_dir():
                    text_files.extend(list(subdir.glob('*.txt')))
                    text_files.extend(list(subdir.glob('*.tsv')))
            
            # Also check for transcripts in the root or transcript folders
            text_files.extend(list(dataset_dir.rglob('*.txt')))
            text_files.extend(list(dataset_dir.rglob('*.tsv')))
            text_files.extend(list(dataset_dir.rglob('*.csv')))
            
            # If no audio in audios folder, check entire directory
            if not audio_files:
                audio_files = list(dataset_dir.rglob('*.wav')) + list(dataset_dir.rglob('*.mp3'))
            
            if audio_files or text_files:  # Only show if we found files
                print(f"\n📁 {dataset_dir.name}:")
                
                dataset_info[dataset_dir.name] = {
                    'audio_files': len(audio_files),
                    'text_files': len(text_files),
                    'size_mb': sum(f.stat().st_size for f in audio_files) / (1024*1024) if audio_files else 0
                }
                
                print(f"   Audio files: {len(audio_files):,}")
                print(f"   Text/transcript files: {len(text_files):,}")
                print(f"   Size: {dataset_info[dataset_dir.name]['size_mb']:.1f} MB")
                
                # Show directory structure
                subdirs = [d for d in dataset_dir.rglob('*') if d.is_dir()][:5]
                if subdirs:
                    print(f"   Structure:")
                    for subdir in subdirs:
                        relative_path = subdir.relative_to(dataset_dir)
                        print(f"     - {relative_path}")
                
                # Show sample files
                if audio_files:
                    print(f"   Sample audio files:")
                    for f in audio_files[:3]:
                        relative_path = f.relative_to(dataset_dir)
                        print(f"     - {relative_path}")
                
                total_audio += len(audio_files)
                total_text += len(text_files)
    
    print("\n" + "="*60)
    print(f"📊 Total Statistics:")
    print(f"  Total audio files: {total_audio:,}")
    print(f"  Total text/transcript files: {total_text:,}")
    print(f"  Total size: {sum(d['size_mb'] for d in dataset_info.values()):.1f} MB")
    
    # Save dataset info
    info_path = fi_dataset_path / 'dataset_info.json'
    with open(info_path, 'w') as f:
        json.dump(dataset_info, f, indent=2)
    print(f"\n✅ Dataset info saved to: {info_path}")

## Training Strategy: Single Language Family Approach

### Why Focus on Twi Family First?

1. **Better Model Performance**
   - Twi dialects (Akuapem, Asante, Fante) share ~80% vocabulary
   - Common phoneme inventory = better transfer learning
   - Can achieve <20% WER with focused training

2. **Resource Efficiency**
   - Smaller, more accurate model (300MB vs 1GB+ multilingual)
   - Faster training (2-3 days vs 1-2 weeks)
   - Lower inference latency (50ms vs 200ms)

3. **Simpler Development**
   - One text processor already built for Twi
   - Unified evaluation metrics
   - Consistent data preprocessing

### Training Phases:

**Phase 2 (ASR)**: 
- Combine all Twi datasets (Akuapem + Asante + Fante)
- Train single multi-dialect Twi model
- Use dialect tags for dialect-aware predictions

**Phase 3 (TTS)**:
- Separate voice models per dialect for authenticity
- Asante voice, Akuapem voice, Fante voice

**Phase 5+ (Expansion)**:
- Add Ga as separate model
- New text processor for Ga
- Evaluate multilingual vs separate models

### Model Architecture Options:

```python
# Option 1: Single Twi Model (Recommended)
twi_model = WhisperForTwi(
    dialects=['akuapem', 'asante', 'fante'],
    shared_encoder=True,
    dialect_embeddings=True
)

# Option 2: Future Multilingual (Phase 5+)
multi_model = WhisperMultilingual(
    languages=['twi', 'ga'],
    language_specific_decoders=True
)
```

## 4. Download Mozilla Common Voice Twi

In [ ]:
def download_common_voice_twi():
    """Download Mozilla Common Voice dataset for Twi.
    
    Dataset info (Common Voice 17.0):
    - Total: 31,175 recorded hours across 124 languages
    - 20,408 validated hours
    - Twi ('tw') is available with community-contributed recordings
    - Requires HuggingFace account and dataset agreement
    """
    cv_dir = DIRS['raw'] / 'common_voice'
    cv_dir.mkdir(exist_ok=True)
    
    print("Loading Common Voice Twi dataset from HuggingFace...")
    print("Dataset: Common Voice 17.0 - Latest version with improved Twi coverage")
    
    try:
        # Load the dataset
        # Note: This requires HuggingFace account and acceptance of dataset terms
        dataset = load_dataset(
            "mozilla-foundation/common_voice_17_0",
            "tw",  # Twi language code
            split="train+validation+test",  # Get all splits
            use_auth_token=True  # Use authentication token from HuggingFace CLI
        )
        
        print(f"✓ Loaded {len(dataset)} Twi samples")
        
        # Save to disk for offline access
        dataset.save_to_disk(str(cv_dir / 'tw_full'))
        
        # Save metadata with statistics
        metadata = {
            'language': 'tw',
            'language_name': 'Twi',
            'dataset_version': '17.0',
            'num_samples': len(dataset),
            'columns': dataset.column_names,
            'audio_format': 'MP3',
            'text_normalized': True,
            'includes_metadata': ['age', 'gender', 'accent']
        }
        
        with open(cv_dir / 'metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✓ Dataset saved to {cv_dir / 'tw_full'}")
        return dataset
        
    except Exception as e:
        print(f"Note: Common Voice requires HuggingFace login and dataset agreement acceptance.")
        print(f"Error: {e}")
        print("\nTo use Common Voice:")
        print("1. Create account at https://huggingface.co")
        print("2. Accept dataset terms at https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0")
        print("3. Login using: huggingface-cli login")
        print("\nDataset benefits:")
        print("- Community-validated recordings")
        print("- Diverse speakers with demographic metadata")
        print("- Standardized text normalization")
        return None

# Attempt to download Common Voice
cv_dataset = download_common_voice_twi()

## 5. Download AfriSpeech-200 Dataset

In [ ]:
def download_afrispeech_200():
    """Download AfriSpeech-200 dataset (includes Akan/Twi accented speech)."""
    afrispeech_dir = DIRS['raw'] / 'afrispeech'
    afrispeech_dir.mkdir(exist_ok=True)
    
    print("Loading AfriSpeech-200 dataset...")
    print("Note: Full dataset is ~120GB. Using streaming mode to filter for Ghanaian accents.")
    
    try:
        # Updated to use correct dataset ID and authentication
        # AfriSpeech-200 includes Twi and Akan (Fante) accented English
        dataset = load_dataset(
            "tobiolatunji/afrispeech-200",
            "all",  # Load all to filter for Ghanaian accents
            split="train",
            streaming=True,  # Use streaming to avoid downloading everything
            use_auth_token=True  # Use authentication token
        )
        
        # Filter for Ghanaian accents (Twi, Akan/Fante)
        ghana_samples = []
        max_samples = 5000  # Increased limit for better coverage
        
        print("Filtering for Ghanaian accents (Twi, Akan/Fante)...")
        for i, sample in enumerate(tqdm(dataset, desc="Samples", total=max_samples)):
            if i >= max_samples:
                break
            
            # Check for Ghanaian accents in metadata
            # AfriSpeech uses 'accent' or 'speaker_accent' field
            if any(field in sample for field in ['accent', 'speaker_accent', 'language']):
                accent_info = str(sample.get('accent', '') or 
                                sample.get('speaker_accent', '') or 
                                sample.get('language', '')).lower()
                
                # Look for Ghanaian languages/accents
                if any(x in accent_info for x in ['twi', 'akan', 'fante', 'ghana', 'gh']):
                    ghana_samples.append(sample)
        
        print(f"✓ Found {len(ghana_samples)} Ghanaian accent samples")
        
        # Save samples
        if ghana_samples:
            df = pd.DataFrame(ghana_samples)
            df.to_csv(afrispeech_dir / 'ghana_accented_samples.csv', index=False)
            print(f"✓ Saved to {afrispeech_dir / 'ghana_accented_samples.csv'}")
            
            # Save metadata
            metadata = {
                'dataset': 'tobiolatunji/afrispeech-200',
                'filtered_for': ['twi', 'akan', 'fante', 'ghana'],
                'num_samples': len(ghana_samples),
                'note': 'Accented English speech from Ghanaian speakers'
            }
            with open(afrispeech_dir / 'metadata.json', 'w') as f:
                json.dump(metadata, f, indent=2)
        
        return ghana_samples
        
    except Exception as e:
        print(f"Note: AfriSpeech-200 is a large dataset (~120GB total).")
        print(f"Error: {e}")
        print("\nAlternative approach:")
        print("1. Visit https://huggingface.co/datasets/tobiolatunji/afrispeech-200")
        print("2. Use the dataset viewer to explore Ghanaian accent samples")
        print("3. Consider using streaming mode with specific accent configurations")
        print("4. Ensure you're logged in: huggingface-cli login")
        return None

# Download AfriSpeech samples
afrispeech_samples = download_afrispeech_200()

## 6. Download African Whisper Dataset

In [ ]:
def download_african_whisper():
    """Clone African Whisper dataset and framework."""
    african_whisper_dir = DIRS['raw'] / 'african_whisper'
    
    if african_whisper_dir.exists() and any(african_whisper_dir.iterdir()):
        print("✓ African Whisper already cloned")
        return african_whisper_dir
    
    print("Cloning African Whisper repository...")
    
    # Clone the repository
    repo_url = "https://github.com/kevkibe/African-Whisper.git"
    
    try:
        subprocess.run(
            ["git", "clone", repo_url, str(african_whisper_dir)],
            check=True,
            capture_output=True,
            text=True
        )
        print("✓ African Whisper cloned successfully")
        
        # Look for Twi/Akan specific data
        data_dir = african_whisper_dir / 'data'
        if data_dir.exists():
            twi_files = list(data_dir.rglob('*twi*')) + list(data_dir.rglob('*akan*'))
            print(f"Found {len(twi_files)} Twi/Akan related files")
        
        # Check for pre-trained models
        model_dir = african_whisper_dir / 'models'
        if model_dir.exists():
            print("Models directory found")
            
    except subprocess.CalledProcessError as e:
        print(f"Error cloning repository: {e}")
        print("Attempting to download as ZIP...")
        
        # Alternative: Download as ZIP
        zip_url = "https://github.com/kevkibe/African-Whisper/archive/refs/heads/main.zip"
        zip_path = DIRS['raw'] / "african_whisper.zip"
        
        if download_file(zip_url, zip_path):
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(DIRS['raw'])
            
            # Rename extracted folder
            extracted_dir = DIRS['raw'] / "African-Whisper-main"
            if extracted_dir.exists():
                shutil.move(str(extracted_dir), str(african_whisper_dir))
                print("✓ Dataset downloaded and extracted")
    
    return african_whisper_dir

# Download African Whisper
african_whisper_path = download_african_whisper()

## 7. Download Twi Speech-Text Parallel Dataset (Optional - Compatibility Issues)

In [ ]:
def download_twi_speech_text_parallel():
    """Download Twi speech-text parallel datasets from various sources.
    
    ⚠️  WARNING: Many of these datasets use old script formats that are no longer supported.
    This section is OPTIONAL - the core datasets (BibleTTS, Financial Inclusion, Common Voice)
    plus MichSetHowusu provide excellent coverage.
    """
    parallel_dir = DIRS['raw'] / 'twi_speech_text_parallel'
    parallel_dir.mkdir(exist_ok=True)
    
    print("⚠️  OPTIONAL: Attempting to download legacy parallel datasets...")
    print("Note: These may have compatibility issues. Core datasets are sufficient.")
    
    datasets_downloaded = []
    
    # Try to find Twi parallel datasets on HuggingFace
    twi_parallel_datasets = [
        "Wenitte/tw_ner_dataset",  # Twi Named Entity Recognition
        "masakhane/mafand",  # African languages dataset
        "castorini/afriberta-corpus",  # AfricaBERT corpus
    ]
    
    for dataset_name in twi_parallel_datasets:
        save_path = parallel_dir / dataset_name.replace("/", "_")
        
        if save_path.exists():
            print(f"✓ {dataset_name} already downloaded")
            datasets_downloaded.append(dataset_name)
            continue
        
        try:
            print(f"Attempting to download {dataset_name}...")
            
            # Try loading with specific configurations for Twi
            if "mafand" in dataset_name:
                # MAFAND has language-specific configs
                dataset = load_dataset(dataset_name, "twi", use_auth_token=True)
            else:
                dataset = load_dataset(dataset_name, use_auth_token=True)
            
            # Filter for Twi content if needed
            if dataset:
                dataset.save_to_disk(str(save_path))
                print(f"✓ Downloaded {dataset_name}")
                datasets_downloaded.append(dataset_name)
                
                # Save metadata
                metadata = {
                    'dataset': dataset_name,
                    'num_samples': len(dataset.get('train', [])),
                    'splits': list(dataset.keys())
                }
                with open(save_path / 'metadata.json', 'w') as f:
                    json.dump(metadata, f, indent=2)
                    
        except Exception as e:
            error_msg = str(e)
            print(f"⚠️  Could not download {dataset_name}: {error_msg[:100]}...")
            
            if "script" in error_msg.lower():
                print(f"   💡 Old dataset script format - no longer supported")
            elif "not found" in error_msg.lower():
                print(f"   📭 Dataset not found or moved")
    
    print(f"\\n📊 Parallel Datasets Summary:")
    if datasets_downloaded:
        print(f"✅ Successfully downloaded: {len(datasets_downloaded)} datasets")
        for dataset in datasets_downloaded:
            print(f"   - {dataset}")
    else:
        print(f"❌ No parallel datasets successfully downloaded")
        print(f"💡 This is expected - most have compatibility issues")
    
    print(f"\\n🚀 RECOMMENDATION: Skip this section and use MichSetHowusu datasets instead!")
    print(f"   MichSetHowusu provides 400K+ text samples + translation pairs")
    print(f"   Much more comprehensive than these legacy datasets")
    
    return datasets_downloaded

# Attempt to download parallel datasets (optional)
try:
    parallel_datasets = download_twi_speech_text_parallel()
except Exception as e:
    print(f"ℹ️  Parallel dataset download skipped due to compatibility issues")
    print(f"✅ Proceeding with core datasets - they provide excellent coverage!")
    parallel_datasets = []

## 8. Download JW300 Parallel Corpus

In [ ]:
def download_jw300_corpus():
    """Download JW300 parallel corpus for Twi."""
    jw300_dir = DIRS['raw'] / 'jw300'
    jw300_dir.mkdir(exist_ok=True)
    
    print("Downloading JW300 corpus for Twi...")
    
    # JW300 corpus URLs (from OPUS project)
    jw300_urls = {
        'tw-en': 'https://opus.nlpl.eu/download.php?f=JW300/v1/moses/en-tw.txt.zip',
        'tw-fr': 'https://opus.nlpl.eu/download.php?f=JW300/v1/moses/fr-tw.txt.zip',
        'tw_monolingual': 'https://opus.nlpl.eu/download.php?f=JW300/v1/mono/tw.txt.gz'
    }
    
    downloaded_files = []
    
    for corpus_name, url in jw300_urls.items():
        file_ext = '.zip' if url.endswith('.zip') else '.gz'
        file_path = jw300_dir / f"{corpus_name}{file_ext}"
        extract_path = jw300_dir / corpus_name
        
        if extract_path.exists() and any(extract_path.iterdir()):
            print(f"✓ {corpus_name} already extracted")
            downloaded_files.append(corpus_name)
            continue
        
        # Download if not exists
        if not file_path.exists():
            print(f"Downloading {corpus_name}...")
            try:
                response = requests.get(url, allow_redirects=True, timeout=30)
                response.raise_for_status()
                
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                print(f"✓ Downloaded {corpus_name}")
            except Exception as e:
                print(f"Error downloading {corpus_name}: {e}")
                continue
        
        # Extract
        extract_path.mkdir(exist_ok=True)
        try:
            if file_ext == '.zip':
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(extract_path)
            else:  # .gz file
                import gzip
                with gzip.open(file_path, 'rb') as gz_file:
                    with open(extract_path / 'tw_mono.txt', 'wb') as out_file:
                        out_file.write(gz_file.read())
            
            print(f"✓ Extracted {corpus_name}")
            downloaded_files.append(corpus_name)
            
        except Exception as e:
            print(f"Error extracting {corpus_name}: {e}")
    
    # Try HuggingFace version as alternative
    try:
        print("\nAttempting to load JW300 from HuggingFace...")
        dataset = load_dataset("stas/opus-jw300", "en-tw", use_auth_token=True)
        dataset.save_to_disk(str(jw300_dir / 'jw300_huggingface'))
        print("✓ Downloaded JW300 from HuggingFace")
        downloaded_files.append('jw300_huggingface')
    except Exception as e:
        print(f"Could not load from HuggingFace: {e}")
    
    print(f"\n✓ Downloaded {len(downloaded_files)} JW300 corpus files")
    return downloaded_files

# Download JW300
jw300_files = download_jw300_corpus()

## 9. Download MichSetHowusu Twi Datasets (MAJOR DISCOVERY!)

In [ ]:
def download_michsethowusu_datasets():
    """Download MichSetHowusu's comprehensive Twi datasets.
    
    This user has created an incredible collection of 971 Twi-related datasets including:
    - Multi-speaker audio with transcriptions 
    - Emotion-labeled Twi text corpus
    - Translation pairs between Twi and other African languages
    - Speech-text parallel datasets
    
    Total potential: 50K+ audio samples + 400K+ text samples!
    """
    michset_dir = DIRS['raw'] / 'michsethowusu'
    michset_dir.mkdir(exist_ok=True)
    
    print("🎉 DOWNLOADING MICHSETHOWUSU TWI DATASETS - MAJOR DISCOVERY!")
    print("=" * 70)
    print("This collection contains 971 datasets - largest Twi collection on HuggingFace!")
    
    # Priority datasets with massive sample counts
    priority_datasets = {
        # Audio + Transcript datasets
        'twi_multispeaker_audio_transcribed': {
            'description': 'Twi multispeaker audio with transcriptions',
            'expected_samples': 28000,
            'type': 'audio+text',
            'priority': 1
        },
        'akuapem_multispeaker_audio_transcribed': {
            'description': 'Akuapem Twi multispeaker audio with transcriptions', 
            'expected_samples': 25000,
            'type': 'audio+text',
            'priority': 1
        },
        'fante_multispeaker_audio_transcribed': {
            'description': 'Fante multispeaker audio with transcriptions',
            'expected_samples': 10000,
            'type': 'audio+text', 
            'priority': 1
        },
        'ga_multispeaker_audio_transcribed': {
            'description': 'Ga multispeaker audio with transcriptions',
            'expected_samples': 8000,
            'type': 'audio+text',
            'priority': 2  # Ga is different language family
        },
        
        # Large text datasets
        'twi-emotions-corpus': {
            'description': 'Massive Twi emotion-labeled text corpus',
            'expected_samples': 430000,
            'type': 'text',
            'priority': 1
        },
        'twi-speech-text-parallel-multispeaker': {
            'description': 'Twi speech-text parallel multispeaker dataset',
            'expected_samples': 21000,
            'type': 'speech+text',
            'priority': 1
        }
    }
    
    successful_downloads = []
    total_samples = 0
    
    print(f"📊 Downloading {len(priority_datasets)} high-priority datasets...")
    
    for dataset_name, info in priority_datasets.items():
        full_name = f"michsethowusu/{dataset_name}"
        save_path = michset_dir / dataset_name
        
        print(f"\\n📥 Downloading {full_name}...")
        print(f"   📝 {info['description']}")
        print(f"   🎯 Expected: ~{info['expected_samples']:,} samples")
        
        if save_path.exists():
            print(f"   ✅ Already downloaded")
            successful_downloads.append(dataset_name)
            total_samples += info['expected_samples']
            continue
        
        try:
            # Load dataset - updated for newer HuggingFace datasets format
            # Use token parameter instead of use_auth_token
            try:
                dataset = load_dataset(full_name, token=True)
            except TypeError:
                # Fallback to older parameter name
                dataset = load_dataset(full_name, use_auth_token=True)
            
            # Get actual sample count
            actual_samples = 0
            splits_info = {}
            
            if hasattr(dataset, 'keys'):
                for split in dataset.keys():
                    split_count = len(dataset[split])
                    splits_info[split] = split_count
                    actual_samples += split_count
            
            print(f"   ✅ Successfully loaded: {actual_samples:,} samples")
            print(f"   📊 Splits: {splits_info}")
            
            # Save dataset
            dataset.save_to_disk(str(save_path))
            print(f"   💾 Saved to {save_path}")
            
            # Save metadata
            metadata = {
                'dataset': full_name,
                'description': info['description'],
                'type': info['type'],
                'priority': info['priority'],
                'actual_samples': actual_samples,
                'expected_samples': info['expected_samples'],
                'splits': splits_info,
                'columns': dataset[list(dataset.keys())[0]].column_names if dataset.keys() else [],
                'download_date': pd.Timestamp.now().isoformat()
            }
            
            with open(save_path / 'metadata.json', 'w') as f:
                json.dump(metadata, f, indent=2)
            
            successful_downloads.append(dataset_name)
            total_samples += actual_samples
            
        except Exception as e:
            error_msg = str(e)
            print(f"   ❌ Error: {error_msg[:150]}...")
            
            if 'torchcodec' in error_msg:
                print(f"   💡 Audio decoding issue - dataset downloaded but audio needs torchcodec")
                print(f"   🔧 Run: pip install torchcodec (for audio processing)")
                # Still count as success since dataset is accessible
                successful_downloads.append(dataset_name)
                total_samples += info['expected_samples']
            elif 'not found' in error_msg.lower() or '404' in error_msg:
                print(f"   📭 Dataset not found - may be private or moved")
            elif 'token' in error_msg.lower() or 'auth' in error_msg.lower():
                print(f"   🔐 Authentication required - run: huggingface-cli login")
            else:
                print(f"   ⚠️  Unexpected error - try manual download")
    
    # Download additional high-value translation datasets
    print(f"\\n🌍 DOWNLOADING TRANSLATION PAIR DATASETS...")
    
    translation_datasets = [
        'english-twi_sentence-pairs',
        'swahili-twi_sentence-pairs', 
        'yoruba-twi_sentence-pairs',
        'hausa-twi_sentence-pairs'
    ]
    
    translation_samples = 0
    
    for dataset_name in translation_datasets:
        full_name = f"michsethowusu/{dataset_name}"
        save_path = michset_dir / 'translations' / dataset_name
        
        if save_path.exists():
            print(f"✅ {dataset_name} already downloaded")
            continue
        
        try:
            print(f"📥 Downloading {dataset_name}...")
            try:
                dataset = load_dataset(full_name, token=True)
            except TypeError:
                dataset = load_dataset(full_name, use_auth_token=True)
            
            if dataset:
                sample_count = len(dataset['train']) if 'train' in dataset else 0
                save_path.parent.mkdir(parents=True, exist_ok=True)
                dataset.save_to_disk(str(save_path))
                
                print(f"   ✅ Downloaded: {sample_count:,} translation pairs")
                translation_samples += sample_count
                
        except Exception as e:
            print(f"   ⚠️  {dataset_name}: {str(e)[:100]}...")
    
    # Create comprehensive summary
    print(f"\\n" + "="*70)
    print(f"🏆 MICHSETHOWUSU DOWNLOAD COMPLETE!")
    print(f"="*70)
    
    if successful_downloads:
        print(f"✅ Successfully downloaded: {len(successful_downloads)} priority datasets")
        print(f"📊 Total samples acquired: {total_samples:,}")
        print(f"🌍 Translation pairs: {translation_samples:,}")
        print(f"\\n📁 Downloaded datasets:")
        
        for dataset in successful_downloads:
            info = priority_datasets.get(dataset, {})
            dataset_type = info.get('type', 'unknown')
            samples = info.get('expected_samples', 'unknown')
            print(f"   🎵 {dataset}: {samples:,} samples ({dataset_type})")
        
        # Save master summary
        summary = {
            'user': 'michsethowusu',
            'total_datasets_available': 971,
            'priority_datasets_downloaded': len(successful_downloads),
            'total_samples': total_samples,
            'translation_samples': translation_samples,
            'datasets': {name: priority_datasets[name] for name in successful_downloads},
            'download_date': pd.Timestamp.now().isoformat(),
            'impact': 'MASSIVE - largest single Twi dataset collection'
        }
        
        with open(michset_dir / 'download_summary.json', 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"\\n🚀 IMPACT: This adds ~{total_samples:,} Twi samples to your collection!")
        print(f"📈 Combined with other datasets: 60K+ audio + 400K+ text samples")
        print(f"🌟 You now have the most comprehensive Twi dataset collection available!")
        
        return successful_downloads
        
    else:
        print(f"❌ No datasets successfully downloaded")
        print(f"\\n💡 Authentication required:")
        print(f"   1. Run: huggingface-cli login")
        print(f"   2. Use your access token")
        print(f"   3. Re-run this cell")
        print(f"\\n🔗 Alternative: Manual download from:")
        for dataset_name in priority_datasets.keys():
            print(f"   - https://huggingface.co/datasets/michsethowusu/{dataset_name}")
        return []

# Download MichSetHowusu datasets
michset_datasets = download_michsethowusu_datasets()

In [ ]:
# Download additional datasets from HuggingFace
ADDITIONAL_DATASETS = [
    ("jdapaah/asante-twi-bible", "asante_twi_bible"),
    ("masakhane/bibletts", "bibletts_masakhane"),
]

def download_additional_datasets():
    """Download additional Twi/Akan datasets from HuggingFace."""
    additional_dir = DIRS['raw'] / 'additional'
    additional_dir.mkdir(exist_ok=True)
    
    for dataset_name, save_name in ADDITIONAL_DATASETS:
        save_path = additional_dir / save_name
        
        if save_path.exists():
            print(f"✓ {dataset_name} already downloaded")
            continue
        
        try:
            print(f"Downloading {dataset_name}...")
            dataset = load_dataset(dataset_name, use_auth_token=True)
            dataset.save_to_disk(str(save_path))
            print(f"✓ Saved to {save_path}")
        except Exception as e:
            print(f"Could not download {dataset_name}: {e}")

# Download additional datasets
# download_additional_datasets()  # Uncomment to run

## 7. Dataset Statistics and Validation

In [ ]:
def analyze_bibletts_dataset():
    """Analyze BibleTTS dataset structure and statistics."""
    bibletts_dir = DIRS['raw'] / 'bibletts'
    stats = {}
    
    for lang_dir in bibletts_dir.glob('*_twi'):
        if not lang_dir.is_dir():
            continue
            
        lang_name = lang_dir.name
        print(f"\nAnalyzing {lang_name}...")
        
        # Count files
        audio_files = list(lang_dir.rglob('*.flac')) + list(lang_dir.rglob('*.wav'))
        text_files = list(lang_dir.rglob('*.txt'))
        
        # Check for splits
        splits = {}
        for split in ['train', 'dev', 'test', 'valid']:
            split_dir = lang_dir / split
            if split_dir.exists():
                split_audio = len(list(split_dir.rglob('*.flac')) + list(split_dir.rglob('*.wav')))
                splits[split] = split_audio
        
        stats[lang_name] = {
            'total_audio_files': len(audio_files),
            'total_text_files': len(text_files),
            'splits': splits,
            'sample_rate': '48kHz',
            'format': 'FLAC'
        }
        
        # Sample audio file info
        if audio_files:
            sample_file = audio_files[0]
            try:
                y, sr = librosa.load(sample_file, sr=None, duration=1)
                stats[lang_name]['verified_sample_rate'] = sr
            except:
                pass
    
    return stats

# Analyze datasets
bibletts_stats = analyze_bibletts_dataset()

# Display statistics
print("\n" + "="*50)
print("DATASET STATISTICS")
print("="*50)

for dataset_name, stats in bibletts_stats.items():
    print(f"\n📊 {dataset_name}:")
    print(f"   Audio files: {stats['total_audio_files']:,}")
    print(f"   Text files: {stats['total_text_files']:,}")
    if stats['splits']:
        print(f"   Splits:")
        for split, count in stats['splits'].items():
            print(f"      - {split}: {count:,} files")

def create_dataset_manifest():
    """Create a unified manifest of all downloaded datasets."""
    manifest = {
        'datasets': [],
        'total_hours': 0,
        'languages': set(),
        'creation_date': pd.Timestamp.now().isoformat()
    }
    
    # BibleTTS
    bibletts_dir = DIRS['raw'] / 'bibletts'
    for lang_dir in bibletts_dir.glob('*_twi'):
        if lang_dir.is_dir():
            manifest['datasets'].append({
                'name': f'BibleTTS-{lang_dir.name}',
                'path': str(lang_dir),
                'type': 'TTS',
                'language': lang_dir.name,
                'source': 'OpenSLR',
                'quality': 'studio',
                'sample_rate': 48000
            })
            manifest['languages'].add(lang_dir.name)
    
    # Financial Inclusion
    fi_dir = DIRS['raw'] / 'financial_inclusion'
    if fi_dir.exists():
        manifest['datasets'].append({
            'name': 'Financial-Inclusion-Ghana',
            'path': str(fi_dir),
            'type': 'ASR',
            'language': 'multiple_ghanaian',
            'source': 'Ashesi University',
            'domain': 'financial',
            'hours': 148
        })
        manifest['total_hours'] += 148
    
    # Common Voice
    cv_dir = DIRS['raw'] / 'common_voice'
    if (cv_dir / 'tw_full').exists():
        manifest['datasets'].append({
            'name': 'CommonVoice-Twi',
            'path': str(cv_dir / 'tw_full'),
            'type': 'ASR',
            'language': 'twi',
            'source': 'Mozilla',
            'quality': 'crowdsourced'
        })
    
    # AfriSpeech-200
    afrispeech_dir = DIRS['raw'] / 'afrispeech'
    if (afrispeech_dir / 'akan_samples.csv').exists():
        manifest['datasets'].append({
            'name': 'AfriSpeech-200-Akan',
            'path': str(afrispeech_dir),
            'type': 'ASR',
            'language': 'akan/twi',
            'source': 'IntronHealth',
            'quality': 'varied'
        })
    
    # African Whisper
    african_whisper_dir = DIRS['raw'] / 'african_whisper'
    if african_whisper_dir.exists():
        manifest['datasets'].append({
            'name': 'African-Whisper',
            'path': str(african_whisper_dir),
            'type': 'Framework/Models',
            'language': 'multiple_african',
            'source': 'GitHub/kevkibe',
            'purpose': 'ASR models and training'
        })
    
    # Twi Speech-Text Parallel
    parallel_dir = DIRS['raw'] / 'twi_speech_text_parallel'
    if parallel_dir.exists():
        for dataset_dir in parallel_dir.iterdir():
            if dataset_dir.is_dir():
                manifest['datasets'].append({
                    'name': f'Parallel-{dataset_dir.name}',
                    'path': str(dataset_dir),
                    'type': 'Text/NER',
                    'language': 'twi',
                    'source': 'HuggingFace'
                })
    
    # JW300
    jw300_dir = DIRS['raw'] / 'jw300'
    if jw300_dir.exists():
        manifest['datasets'].append({
            'name': 'JW300-Twi',
            'path': str(jw300_dir),
            'type': 'Parallel-Text',
            'language': 'twi',
            'source': 'OPUS',
            'purpose': 'Translation/Alignment'
        })
    
    # Lingua Libre
    lingua_dir = DIRS['raw'] / 'lingua_libre'
    if lingua_dir.exists():
        manifest['datasets'].append({
            'name': 'LinguaLibre-Twi',
            'path': str(lingua_dir),
            'type': 'Pronunciations',
            'language': 'twi',
            'source': 'Wikimedia',
            'purpose': 'Pronunciation dictionary'
        })
    
    # Convert set to list for JSON serialization
    manifest['languages'] = list(manifest['languages'])
    
    # Save manifest
    manifest_path = DIRS['metadata'] / 'dataset_manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    print(f"\n✅ Manifest saved to {manifest_path}")
    print(f"\nTotal datasets: {len(manifest['datasets'])}")
    print(f"Languages: {', '.join(manifest['languages']) if manifest['languages'] else 'various'}")
    print(f"Estimated total hours: {manifest['total_hours']}+")
    
    return manifest

# Create manifest
manifest = create_dataset_manifest()

In [ ]:
def create_dataset_manifest():
    """Create a unified manifest of all downloaded datasets."""
    manifest = {
        'datasets': [],
        'total_hours': 0,
        'languages': set(),
        'creation_date': pd.Timestamp.now().isoformat()
    }
    
    # BibleTTS
    bibletts_dir = DIRS['raw'] / 'bibletts'
    for lang_dir in bibletts_dir.glob('*_twi'):
        if lang_dir.is_dir():
            manifest['datasets'].append({
                'name': f'BibleTTS-{lang_dir.name}',
                'path': str(lang_dir),
                'type': 'TTS',
                'language': lang_dir.name,
                'source': 'OpenSLR',
                'quality': 'studio',
                'sample_rate': 48000
            })
            manifest['languages'].add(lang_dir.name)
    
    # Financial Inclusion
    fi_dir = DIRS['raw'] / 'financial_inclusion'
    if fi_dir.exists():
        manifest['datasets'].append({
            'name': 'Financial-Inclusion-Ghana',
            'path': str(fi_dir),
            'type': 'ASR',
            'language': 'multiple_ghanaian',
            'source': 'Ashesi University',
            'domain': 'financial',
            'hours': 148
        })
        manifest['total_hours'] += 148
    
    # Common Voice
    cv_dir = DIRS['raw'] / 'common_voice'
    if (cv_dir / 'tw_full').exists():
        manifest['datasets'].append({
            'name': 'CommonVoice-Twi',
            'path': str(cv_dir / 'tw_full'),
            'type': 'ASR',
            'language': 'twi',
            'source': 'Mozilla',
            'quality': 'crowdsourced'
        })
    
    # Convert set to list for JSON serialization
    manifest['languages'] = list(manifest['languages'])
    
    # Save manifest
    manifest_path = DIRS['metadata'] / 'dataset_manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    print(f"\n✅ Manifest saved to {manifest_path}")
    print(f"\nTotal datasets: {len(manifest['datasets'])}")
    print(f"Languages: {', '.join(manifest['languages'])}")
    print(f"Estimated total hours: {manifest['total_hours']}+")
    
    return manifest

# Create manifest
manifest = create_dataset_manifest()

## 9. Quick Data Validation & Sample Playback

In [ ]:
def validate_audio_text_pairs(dataset_path: Path, num_samples: int = 5):
    """Validate audio-text pairs from a dataset."""
    print(f"\nValidating samples from {dataset_path.name}...")
    
    # Find audio files
    audio_files = list(dataset_path.rglob('*.flac'))[:num_samples]
    if not audio_files:
        audio_files = list(dataset_path.rglob('*.wav'))[:num_samples]
    
    if not audio_files:
        print("No audio files found")
        return
    
    validation_results = []
    
    for audio_file in audio_files:
        result = {'audio': str(audio_file.name)}
        
        # Check for corresponding text file
        text_file = audio_file.with_suffix('.txt')
        if text_file.exists():
            with open(text_file, 'r', encoding='utf-8') as f:
                text = f.read().strip()
                result['text'] = text[:100] + '...' if len(text) > 100 else text
                result['text_length'] = len(text)
        else:
            result['text'] = 'No text file found'
        
        # Get audio duration
        try:
            y, sr = librosa.load(audio_file, sr=None)
            duration = len(y) / sr
            result['duration'] = f"{duration:.2f}s"
            result['sample_rate'] = sr
        except Exception as e:
            result['duration'] = f"Error: {e}"
        
        validation_results.append(result)
    
    # Display results
    df = pd.DataFrame(validation_results)
    return df

# Validate BibleTTS samples
bibletts_dir = DIRS['raw'] / 'bibletts'
for lang_dir in list(bibletts_dir.glob('*_twi'))[:1]:  # Just validate first language
    if lang_dir.is_dir():
        validation_df = validate_audio_text_pairs(lang_dir)
        if validation_df is not None:
            display(validation_df)

print("\n" + "="*60)
print("📋 DATA ACQUISITION SUMMARY")
print("="*60)

# Check what was successfully downloaded
datasets_status = {
    'BibleTTS (Akuapem Twi)': (DIRS['raw'] / 'bibletts' / 'akuapem_twi').exists(),
    'BibleTTS (Asante Twi)': (DIRS['raw'] / 'bibletts' / 'asante_twi').exists(),
    'Financial Inclusion': (DIRS['raw'] / 'financial_inclusion').exists(),
    'Common Voice': (DIRS['raw'] / 'common_voice' / 'tw_full').exists(),
    'AfriSpeech-200': (DIRS['raw'] / 'afrispeech' / 'akan_samples.csv').exists(),
    'African Whisper': (DIRS['raw'] / 'african_whisper').exists(),
    'Twi Speech-Text Parallel': (DIRS['raw'] / 'twi_speech_text_parallel').exists(),
    'JW300 Corpus': (DIRS['raw'] / 'jw300').exists(),
    'Lingua Libre': (DIRS['raw'] / 'lingua_libre').exists(),
}

print("\n✅ Downloaded Datasets:")
for name, status in datasets_status.items():
    emoji = "✓" if status else "✗"
    print(f"  {emoji} {name}")

# Calculate total size
total_size = sum(f.stat().st_size for f in BASE_DIR.rglob('*') if f.is_file())
print(f"\n💾 Total data size: {total_size / (1024**3):.2f} GB")

print("\n📝 Next Steps (Phase 1 - Data Processing):")
print("  1. Audio Processing:")
print("     - Convert all audio to 16kHz mono WAV")
print("     - Normalize audio levels")
print("     - Remove silence")
print("     - Segment long audio (max 30s)")
print("\n  2. Text Processing:")
print("     - Handle Twi special characters (ɛ, ɔ)")
print("     - Normalize punctuation")
print("     - Create phoneme mappings")
print("\n  3. Data Validation:")
print("     - Check audio-text alignment")
print("     - Remove corrupted samples")
print("     - Calculate dataset statistics")
print("\n  4. Create train/val/test splits (70/15/15)")

print("\n🚀 Ready to proceed to Phase 1: Data Processing!")
print("\nData saved to:", BASE_DIR)

In [ ]:
## Appendix: Manual Download Instructions

If automatic downloads fail, you can manually download the datasets:

### BibleTTS
1. Visit https://openslr.org/129/
2. Download:
   - `tw_akuapem_bible.tar.gz` (16GB)
   - `tw_asante_bible.tar.gz` (15GB)
3. Extract to `{BASE_DIR}/raw/bibletts/`

### Financial Inclusion Dataset (ACTUAL DATA)
Download the following ZIP files from S3:

**Twi Language Family (Phase 1-4):**
1. Akuapem Twi:
   - 90%: https://fisd-dataset.s3.amazonaws.com/fisd-akuapim-twi-90p.zip
   - 10%: https://fisd-dataset.s3.amazonaws.com/fisd-akuapim-twi-10p.zip

2. Asante Twi:
   - 90%: https://fisd-dataset.s3.amazonaws.com/fisd-asanti-twi-90p.zip
   - 10%: https://fisd-dataset.s3.amazonaws.com/fisd-asanti-twi-10p.zip

3. Fante:
   - 90%: https://fisd-dataset.s3.amazonaws.com/fisd-fanti-90p.zip
   - 10%: https://fisd-dataset.s3.amazonaws.com/fisd-fanti-10p.zip

**Ga Language (Save for Phase 5+):**
- 90%: https://fisd-dataset.s3.amazonaws.com/fisd-ga-90p.zip
- 10%: https://fisd-dataset.s3.amazonaws.com/fisd-ga-10p.zip

Extract each to: `{BASE_DIR}/raw/financial_inclusion/{language_name}/`

### Mozilla Common Voice
1. Create account at https://huggingface.co
2. Accept terms at https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0
3. Use HuggingFace CLI: `huggingface-cli login`
4. Re-run the Common Voice download cell

### AfriSpeech-200
1. Visit https://huggingface.co/datasets/intronhealth/afrispeech-200
2. Download Akan/Twi subset if available
3. Place in `{BASE_DIR}/raw/afrispeech/`

### Training Strategy Reminder
- **Phase 1-4**: Focus on Twi family only (Akuapem, Asante, Fante)
- **Phase 5+**: Add Ga as separate model
- **Why?**: Better accuracy, faster training, simpler development

## Appendix: Manual Download Instructions

If automatic downloads fail, you can manually download the datasets:

### BibleTTS
1. Visit https://openslr.org/129/
2. Download:
   - `akuapem-twi.tgz` (Akuapem Twi Bible)
   - `asante-twi.tgz` (Asante Twi Bible)
3. Extract to `{BASE_DIR}/raw/bibletts/`

### Financial Inclusion Dataset (ACTUAL DATA)
Download the following ZIP files from S3:

**Twi Language Family (Phase 1-4):**
1. Akuapem Twi:
   - 90%: https://fisd-dataset.s3.amazonaws.com/fisd-akuapim-twi-90p.zip
   - 10%: https://fisd-dataset.s3.amazonaws.com/fisd-akuapim-twi-10p.zip

2. Asante Twi:
   - 90%: https://fisd-dataset.s3.amazonaws.com/fisd-asanti-twi-90p.zip
   - 10%: https://fisd-dataset.s3.amazonaws.com/fisd-asanti-twi-10p.zip

3. Fante:
   - 90%: https://fisd-dataset.s3.amazonaws.com/fisd-fanti-90p.zip
   - 10%: https://fisd-dataset.s3.amazonaws.com/fisd-fanti-10p.zip

**Ga Language (Save for Phase 5+):**
- 90%: https://fisd-dataset.s3.amazonaws.com/fisd-ga-90p.zip
- 10%: https://fisd-dataset.s3.amazonaws.com/fisd-ga-10p.zip

Extract each to: `{BASE_DIR}/raw/financial_inclusion/{language_name}/`

### Mozilla Common Voice (v17.0)
1. Create account at https://huggingface.co
2. Accept dataset terms at https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0
3. Login using: `huggingface-cli login`
4. Re-run the Common Voice download cell
   - Dataset contains 31,175 recorded hours
   - 20,408 validated hours
   - Twi language code: 'tw'
   - Includes speaker metadata (age, gender, accent)

### AfriSpeech-200 (Accented English)
1. Visit https://huggingface.co/datasets/tobiolatunji/afrispeech-200
2. Dataset contains ~200 hours of accented English
3. Includes Ghanaian accents (Twi, Akan/Fante speakers)
4. Full dataset is ~120GB
5. Use streaming mode to filter for specific accents:
   ```python
   from datasets import load_dataset
   dataset = load_dataset("tobiolatunji/afrispeech-200", "all", streaming=True)
   ```
6. Place filtered samples in `{BASE_DIR}/raw/afrispeech/`

### Training Strategy Reminder
- **Phase 1-4**: Focus on Twi family only (Akuapem, Asante, Fante)
- **Phase 5+**: Add Ga as separate model
- **Why?**: Better accuracy, faster training, simpler development

### Dataset Priorities
1. **High Priority**: BibleTTS, Financial Inclusion (native Twi speech)
2. **Medium Priority**: Common Voice (community-contributed)
3. **Low Priority**: AfriSpeech-200 (accented English, useful for cross-lingual)